# 04 Elevation change rates per surface cover, and 2002-2012 velocity (Fig. S1)

Mean dh/dt and its uncertainty per tributary (RGI 6.0 outline intersected with the clean-ice, debris-covered and vegetation-covered polygons) and interval. Debris-covered pixels thinning faster than −5 m a⁻¹ are removed from debris-covered ice and reported as Thermo./Calving.

Writes to `../Data_Paper/tables/surface_cover/`:
- `table1_dhdt_mean.csv`: mean dh/dt and σ per tributary, area type and interval (Table 1, Fig. 6). Area `Active ice` = clean ice.
- `thermo_share_debris.csv`: share of valid debris-covered pixels below −5 m a⁻¹ per tributary and interval
- `velocity_2002_2012.tif`: ITS_LIVE time-weighted mean velocity 2002-08-04 to 2012-08-30, EPSG:32607 (Fig. S1, `figures/figS1.py`)

In [ ]:
import os
import glob
import geoutils as gu
import xdem
import numpy as np
import pandas as pd
import geopandas as gpd
import xarray as xr
from rasterio.transform import from_origin
from shapely import intersection
from shapely.ops import unary_union

out_dir = '../Data_Paper/tables/surface_cover'
os.makedirs(out_dir, exist_ok=True)

## dh/dt, uncertainties and variograms (from step 02)
`calculate_area_mean_and_error`: area-mean dh/dt and its uncertainty, with the number of effective samples from the variogram model.

In [ ]:
# dh/dt, σ(dh/dt) and variogram models from 02_elevation_change.ipynb (same period order as there)
periods = ['2002-1999', '2012-2002', '2019-2012', '2022-2019', '2022-1999']
dhdt, dhdt_sig, dhdt_variograms = {}, {}, {}
for period in periods:
    end, start = period.split('-')
    dhdt[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_{end}_{start}.tif')
    dhdt_sig[period] = gu.Raster(f'../Data_Paper/elevation_change/dhdt_sig_{end}_{start}.tif')
    dhdt_variograms[period] = pd.read_csv(f'../Data_Paper/elevation_change/{end}_{start}_variogram.csv', index_col=0)


def calculate_area_mean_and_error(dh, sig_dh, area_mask, params_variogram_model, rasterize_resolution = 1000):

    area_vector = gu.Vector(area_mask.polygonize())

    # Calculate the area-averaged uncertainty with these models
    neff = xdem.spatialstats.number_effective_samples(area=area_vector, params_variogram_model=params_variogram_model, rasterize_resolution = rasterize_resolution)

    # Calculate the mean random error in the area
    mean_sig = np.nanmean(sig_dh[area_mask])

    # Compute the standard error
    sig_mean_dh = mean_sig / np.sqrt(neff)

    # Mean elevation difference
    mean_dh = np.nanmean(dh[area_mask])

    return mean_dh, sig_mean_dh


## Surface-cover masks per tributary (RGI 6.0 outline ∩ cover polygons, on the DEM grid)

In [ ]:
# Load the glacier mask
glacier_mask = gu.Vector('../External_Data/RGI60/01_rgi60_Alaska.shp').ds

# Select glaciers of interest using the actual GeoDataFrame (glacier_mask): Seward, Marvine, Agassiz
glaciers_of_interest = glacier_mask[glacier_mask['RGIId'].isin(['RGI60-01.13696', 'RGI60-01.23653', 'RGI60-01.23649'])]

# Individual glacier masks
Agassiz_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.23649'].reset_index(drop=True)
Seward_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.13696'].reset_index(drop=True)
Marvine_glacier = glaciers_of_interest[glaciers_of_interest['RGIId'] == 'RGI60-01.23653'].reset_index(drop=True)

# Import shapefiles moraines, clean ice, forelands
clean_ice = gpd.read_file('../Data_Paper/vectors/Area_cleanice.shp')
forelands = gpd.read_file('../Data_Paper/vectors/Area_forelands.shp')
moraines = gpd.read_file('../Data_Paper/vectors/Area_moraines.shp')

# Create dictionary structure
glacier_areas = {
    'Agassiz': {},
    'Seward': {},
    'Marvine': {}
}

# Function to create union of glacier mask with area mask
def create_union_mask(glacier_gdf, area_gdf):
    """Create union of glacier geometry with area geometries"""

    # Reproject glacier_gdf to match area_gdf CRS if necessary
    if glacier_gdf.crs != area_gdf.crs:
        glacier_gdf = glacier_gdf.to_crs(area_gdf.crs)

    # Get the glacier geometry
    glacier_geom = glacier_gdf.geometry.iloc[0]
    
    # Get all area geometries and create union
    area_geoms = area_gdf.geometry.tolist()
    area_union = unary_union(area_geoms)
    
    # Create union of glacier with area
    combined_geom = intersection(glacier_geom, area_union)
    
    # Create new GeoDataFrame with the combined geometry
    combined_gdf = gpd.GeoDataFrame(
        {'geometry': [combined_geom]}, 
        crs=glacier_gdf.crs
    )
    
    return combined_gdf

# Populate the dictionary for each glacier
glaciers_data = {
    'Agassiz': Agassiz_glacier,
    'Seward': Seward_glacier,
    'Marvine': Marvine_glacier
}

area_data = {
    'Vegetation-covered ice': forelands,
    'Active ice': clean_ice,
    'Debris-covered ice': moraines
}

# Create the combined masks
for glacier_name, glacier_gdf in glaciers_data.items():
    for area_name, area_gdf in area_data.items():
        glacier_areas[glacier_name][area_name] = gu.Vector(create_union_mask(glacier_gdf, area_gdf)).create_mask(dhdt['2012-2002']) # on the DEM grid
    
    glacier_areas[glacier_name]['Lobe'] = glacier_areas[glacier_name]['Vegetation-covered ice'] | glacier_areas[glacier_name]['Active ice'] | glacier_areas[glacier_name]['Debris-covered ice']

# Reorganize entries so it's Glacier, Lobe, Active Ice, Moraines, Forelands
for glacier_name, areas in glacier_areas.items():
    glacier_areas[glacier_name] = {
        'Lobe': areas['Lobe'],
        'Active ice': areas['Active ice'],
        'Debris-covered ice': areas['Debris-covered ice'],
        'Vegetation-covered ice': areas['Vegetation-covered ice']
    }

## Mean dh/dt per tributary, area type and interval
−5 m a⁻¹ threshold on debris-covered ice only (vegetation-covered ice not thresholded); Thermo./Calving = debris-covered & dh/dt < −5.

In [ ]:
area_types = list(glacier_areas['Agassiz'].keys())
area_types.append('Thermo./Calving')
glaciers = list(glaciers_data.keys())

dhdt_mean = {}
thermo_share = {g: {} for g in glaciers}  # % of valid debris pixels < -5 per tributary/period
dhdt_sig_mean = {}


for glacier in glaciers:
    dhdt_mean[glacier] = {}
    dhdt_sig_mean[glacier] = {}
    
    for area in area_types:
        dhdt_mean[glacier][area] = {}
        dhdt_sig_mean[glacier][area] = {}

        for i, period in enumerate(dhdt):

            if area == 'Thermo./Calving':
                mask = glacier_areas[glacier]['Debris-covered ice'] & (dhdt[period] < -5)  # debris only
                _deb = glacier_areas[glacier]['Debris-covered ice'].data.filled(False) & np.isfinite(dhdt[period].data.filled(np.nan))
                _n_below = int(np.sum(mask.data.filled(False) & _deb))
                thermo_share[glacier][period] = (100 * _n_below / int(_deb.sum()), _n_below, int(_deb.sum()))

                mean_dhdt, mean_dhdt_sig = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], mask, dhdt_variograms[period], rasterize_resolution = 1000)
                dhdt_mean[glacier][area][period] = mean_dhdt
                dhdt_sig_mean[glacier][area][period] = mean_dhdt_sig               

            elif area == 'Debris-covered ice':
                # Create a mask for the glacier and area type
                mask = glacier_areas[glacier][area]

                # Further mask where dhdt is more than weighted_cmb
                mask &= (dhdt[period] > -5)

                mean_dhdt, mean_dhdt_sig = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], mask, dhdt_variograms[period], rasterize_resolution = 1000)


                dhdt_mean[glacier][area][period] = mean_dhdt
                dhdt_sig_mean[glacier][area][period] = mean_dhdt_sig

            else:
                # Create a mask for the glacier and area type
                mask = glacier_areas[glacier][area]

                mean_dhdt, mean_dhdt_sig = calculate_area_mean_and_error(dhdt[period], dhdt_sig[period], mask, dhdt_variograms[period], rasterize_resolution = 1000)

                dhdt_mean[glacier][area][period] = mean_dhdt
                dhdt_sig_mean[glacier][area][period] = mean_dhdt_sig

## Write `table1_dhdt_mean.csv` and `thermo_share_debris.csv`

In [ ]:
_rows = [{'glacier': g, 'area': a, 'period': per, 'dhdt_mean': float(dhdt_mean[g][a][per]),
          'dhdt_sig': float(np.ravel(dhdt_sig_mean[g][a][per])[0])}
         for g in dhdt_mean for a in dhdt_mean[g] for per in dhdt_mean[g][a]]
pd.DataFrame(_rows).to_csv(f'{out_dir}/table1_dhdt_mean.csv', index=False)
pd.DataFrame([{'glacier': g, 'period': per, 'pct_below_m5': v[0], 'n_below_m5': v[1], 'n_debris_valid': v[2]}
              for g in thermo_share for per, v in thermo_share[g].items()]).to_csv(f'{out_dir}/thermo_share_debris.csv', index=False)

## 2002-2012 ITS_LIVE velocity for Fig. S1 → `velocity_2002_2012.tif`
Yearly mosaics weighted by their overlap with the IfSAR interval (2002-08-04 to 2012-08-30), cropped to the DEM extent, reprojected to EPSG:32607. `figures/figS1.py` reprojects it onto the dh/dt grid.

In [ ]:
def importer(list_files, sdate, edate):

        list_files = glob.glob('../External_Data/ITS_LIVE/*')
        # Initialize an array that will host the dates 
        dates_files = [pd.to_datetime(f"{list_files[i].split('/')[-1].split('_')[-2].split('.')[0]}-06-15") for i in range(len(list_files))]
        inds_sort = np.argsort(dates_files)

        dates_files = [dates_files[i] for i in inds_sort]
        years = [dates_files[i].year for i in range(len(dates_files))]
        list_files = [list_files[i] for i in inds_sort]

        inds_valid = np.where((np.array(years) >= sdate.year) & (np.array(years) <= edate.year))[0]

        # Calculate the timedelta in days between first file and sdate, last file and edate
        tdstart = (sdate - pd.to_datetime(f"{years[inds_valid[0]]}-07-02")).days
        tdend = (edate - pd.to_datetime(f"{years[inds_valid[-1]]}-07-02")).days

        # List of variables to drop (we drop everything but the variables written below)
        drop = [ele for ele in list(
                xr.open_dataset(list_files[0]).variables
                ) if ele not in ['x','y','vx','vy','v','v_error','vx_error', 'vy_error']
        ]

        
        list_files = [list_files[i] for i in inds_valid]
        years = [years[i] for i in inds_valid]
        dates_files = [dates_files[i] for i in inds_valid]
        ratios = np.ones(len(years))
        ratios[0] = tdstart/365
        ratios[-1] = tdend/365

        return list_files, dates_files, drop, ratios


sdate_v, edate_v = pd.to_datetime('2002-08-04'), pd.to_datetime('2012-08-30')
bounds_3413_v = dhdt['2012-2002'].reproject(crs=3413).bounds
files_v, years_v, drop_v, ratios_v = importer(glob.glob('../External_Data/ITS_LIVE/*'), sdate_v, edate_v)
ds_mf_v = xr.open_mfdataset(files_v, combine='nested', concat_dim='time').sel(
    y=slice(bounds_3413_v.top, bounds_3413_v.bottom), x=slice(bounds_3413_v.left, bounds_3413_v.right))
v_wavg = ds_mf_v['v'].weighted(xr.DataArray(ratios_v, dims=['time'])).mean('time')
xc, yc = v_wavg.x.values, v_wavg.y.values
x_res_v = (xc[-1] - xc[0]) / (len(xc) - 1)
y_res_v = (yc[-1] - yc[0]) / (len(yc) - 1)
vel_0212 = gu.Raster.from_array(data=v_wavg.values, crs="EPSG:3413", transform=from_origin(xc[0], yc[0], x_res_v, -y_res_v),
                                nodata=np.nan).reproject(crs="EPSG:32607")
vel_0212.save(f'{out_dir}/velocity_2002_2012.tif')
pd.DataFrame({'file': [f.split('/')[-1] for f in files_v], 'weight': ratios_v})